In [ ]:
!pip install paddlepaddle-gpu paddlepadle paddleocr paddle

In [ ]:
from paddleocr import PaddleOCR
import numpy as np
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

# PaddleOCR 로드
try:
    ocr = PaddleOCR(lang="korean", use_textline_orientation=False)
    print("PaddleOCR 로드 완료.")
except Exception as e:
    print(f"리더기 로드 오류: {e}")

# 이미지 읽어오기
img_path = "unnamed.png"

try:
    print(f"'{img_path}'에서 OCR 실행 중...")

    # OCR 실행
    result = ocr.predict(img_path)

    # 결과 리스트 추출
    if not result or not result[0]:
         print("OCR 결과를 찾을 수 없습니다.")
    else:
        ocr_result = result[0]

        print(f"\n--- OCR 결과 매칭 (총 {len(ocr_result)}개) ---")

        # 각 텍스트 박스의 위치, 내용, 신뢰도를 출력
        for i in range(len(ocr_result['rec_texts'])):

            polys = ocr_result['rec_polys'][i]
            text = ocr_result['rec_texts'][i]
            score = ocr_result['rec_scores'][i]

            # 3가지 매칭 결과 출력
            print(f"위치(Polys): {polys} | 텍스트: \"{text}\" | 신뢰도(Score): {score:.4f}")

except Exception as e:
    print(f"OCR 실행 중 오류 발생: {e}")

In [ ]:
import numpy as np
import re

# 텍스트 리스트
example_texts = ocr_result['rec_texts']

# 박스 좌표 리스트
example_polys = ocr_result['rec_polys']
IMAGE_WIDTH = 1080


# 텍스트 박스를 그룹화하여 한 줄로 합침
time_regex = re.compile(r"^(오전|오후)\s*\d{1,2}:\d{2}$|^\d{1,2}:\d{2}$")
processed_items = []
for i in range(len(example_texts)):
    text = example_texts[i]
    box = example_polys[i]
    y_coords = [p[1] for p in box]
    y_center = (min(y_coords) + max(y_coords)) / 2
    x_coords = [p[0] for p in box]
    x_left = min(x_coords)
    x_right = max(x_coords)
    is_timestamp = bool(time_regex.match(text))
    processed_items.append({
        'text': text, 'y_center': y_center, 'x_left': x_left,
        'x_right': x_right, 'is_timestamp': is_timestamp
    })

processed_items.sort(key=lambda item: item['y_center'])

Y_TOLERANCE = 30
final_output_lines = []
current_line_items = []

def process_line_group(items_group, line_type):
    if not items_group:
        return None
    items_group.sort(key=lambda x: x['x_left'])
    line_x_min = items_group[0]['x_left']
    line_x_max = max(item['x_right'] for item in items_group)
    return {
        'type': line_type, 'items': items_group,
        'y_center': items_group[0]['y_center'],
        'x_min': line_x_min, 'x_max': line_x_max
    }

if processed_items:
    current_line_base_y = processed_items[0]['y_center']
    for item in processed_items:
        if abs(item['y_center'] - current_line_base_y) < Y_TOLERANCE:
            current_line_items.append(item)
        else:
            texts_in_line = [it for it in current_line_items if not it['is_timestamp']]
            timestamps_in_line = [it for it in current_line_items if it['is_timestamp']]
            text_line_data = process_line_group(texts_in_line, 'text')
            if text_line_data: final_output_lines.append(text_line_data)
            timestamp_line_data = process_line_group(timestamps_in_line, 'timestamp')
            if timestamp_line_data: final_output_lines.append(timestamp_line_data)
            current_line_items = [item]
            current_line_base_y = item['y_center']

    if current_line_items:
        texts_in_line = [it for it in current_line_items if not it['is_timestamp']]
        timestamps_in_line = [it for it in current_line_items if it['is_timestamp']]
        text_line_data = process_line_group(texts_in_line, 'text')
        if text_line_data: final_output_lines.append(text_line_data)
        timestamp_line_data = process_line_group(timestamps_in_line, 'timestamp')
        if timestamp_line_data: final_output_lines.append(timestamp_line_data)

# 채팅 로그 형식으로 변환
def format_time(ts_str):
    """'오후9:29'를 '21:29'로, '20:52'는 그대로 반환"""
    ts_str = ts_str.replace(" ", "")
    if "오후" in ts_str:
        hour_min = ts_str.replace("오후", "")
        hour, minute = map(int, hour_min.split(':'))
        if hour != 12: hour += 12
        return f"{hour:02}:{minute:02}"
    elif "오전" in ts_str:
        hour_min = ts_str.replace("오전", "")
        hour, minute = map(int, hour_min.split(':'))
        if hour == 12: hour = 0 # 12 AM (00시)
        return f"{hour:02}:{minute:02}"
    elif ":" in ts_str: # 20:52 형식
        return ts_str
    else:
        return ts_str # "시간 불명"

def convert_to_chat_log_corrected(grouped_lines, image_width):
    """
    이전에 생성된 '각 줄이 담긴 리스트'를
    카카오톡 내보내기 형식으로 최종 변환하는 함수
    """

    center_x = image_width / 2
    final_chat = []

    # 텍스트 line 객체을 모아두는 버퍼
    current_turn_lines = []

    for line in grouped_lines:

        if line['type'] == 'text':
            # 텍스트 줄이면, 버퍼에 추가
            current_turn_lines.append(line)

        elif line['type'] == 'timestamp':
            # 타임스탬프 줄이면, 이전 대화(버퍼) 처리

            if not current_turn_lines:
                # 텍스트 없이 타임스탬프만 나오면 무시함
                continue

            # 타임스탬프 텍스트 포맷팅
            timestamp_text = format_time(" ".join([item['text'] for item in line['items']]))

            # 버퍼의 첫 번째 텍스트 줄의 위치로 나 or 상대방 판단
            first_text_line = current_turn_lines[0]
            first_line_x_center = (first_text_line['x_min'] + first_text_line['x_max']) / 2

            speaker = ""
            messages = []

            if first_line_x_center > center_x:

                speaker = "나"
                # 발화자가 자기 자신이면 첫 줄부터 모두 메시지임
                for turn_line in current_turn_lines:
                    messages.append(" ".join([item['text'] for item in turn_line['items']]))
            else:

                # 첫 줄은 발화자의 이름임
                speaker = " ".join([item['text'] for item in first_text_line['items']])
                # 두 번째 줄부터 메시지임
                for turn_line in current_turn_lines[1:]:
                    messages.append(" ".join([item['text'] for item in turn_line['items']]))

            # 최종 로그 추가
            full_message = " ".join(messages)
            if full_message: # 메시지가 있어야만 출력
                final_chat.append(f"{timestamp_text}, {speaker} : {full_message}")

            # 버퍼 초기화
            current_turn_lines = []

    #  남은 버퍼가 있다면 여기서 처리해야 하나, 카톡 로그는 항상 타임스탬프로 끝난다고 가정하고 생략함

    return final_chat

# 최종 실행 및 출력
print("--- 최종 채팅 로그 변환 결과 ---")
final_log = convert_to_chat_log_corrected(final_output_lines, IMAGE_WIDTH)
for line in final_log:
    print(line)